# Lab 1: Local Inference & the OpenAI Protocol

Track: **Python**. Model: **qwen2.5:1.5b** running locally in **Ollama**.

This notebook walks through the lab in order and shows every output.
The real deliverable (the terminal chat) is `chat_loop.py`.

## 1. Is Ollama running, and is the model there?

Ollama is a small server on `localhost:11434`. `/api/tags` lists the downloaded models.

In [1]:
import json, urllib.request

with urllib.request.urlopen("http://localhost:11434/api/tags") as r:
    models = json.load(r)["models"]
print("Models on this machine:", [m["name"] for m in models])

Models on this machine: ['qwen2.5:1.5b']


## 2. The raw HTTP request (same as the `curl` in the lab)

No library: just a JSON `POST` to `/v1/chat/completions`. This is all the SDK does for us.

In [2]:
payload = {
    "model": "qwen2.5:1.5b",
    "messages": [
        {"role": "system", "content": "You are a concise Unix system administrator."},
        {"role": "user", "content": "What command lists all TCP listening sockets?"},
    ],
    "temperature": 0.0,
}
request = urllib.request.Request(
    "http://localhost:11434/v1/chat/completions",
    data=json.dumps(payload).encode(),
    headers={"Content-Type": "application/json"},
)
with urllib.request.urlopen(request) as r:
    data = json.load(r)

print("choices[0].message.content:\n")
print(data["choices"][0]["message"]["content"])
print("\nusage.prompt_tokens     =", data["usage"]["prompt_tokens"])
print("usage.completion_tokens =", data["usage"]["completion_tokens"])

choices[0].message.content:

The command to list all TCP listening sockets is `netstat` or `ss` with the `-t` option. Here's how you can use it:

### Using `netstat`
```bash
netstat -t
```

### Using `ss`
```bash
ss -t
```

### Explanation
- `netstat`: A command-line utility that displays network connections, routing tables, interface statistics, and multicast memberships.
- `-t`: Specifies that the output should be in TCP format.

### What `netstat` Lists
- **Connections**: Shows active TCP connections.
- **Routes**: Displays routing tables.
- **Interfaces**: Lists network interfaces.
- **Multicast**: Shows multicast memberships.
- **Statistics**: Displays statistics for each network interface.

### What `ss` Lists
- **TCP**: Lists all TCP connections and listening sockets.
- **UDP**: Lists all UDP connections and listening sockets.
- **All**: Lists all connections, routes, interfaces, and statistics.

### Example Output
```bash
tcp        0      0 0.0.0.0:22         0.0.0.0:*       L

## 3. Same thing with the `openai` Python package (`client_test.py`)

Only `base_url` points to Ollama. The `api_key` is required by the package but Ollama ignores it.

In [3]:
import client_test

client_test.run_single_turn()

Model Output:

The primary difference between a process and a thread is that a process is an independent instance of a program, while a thread is an execution unit within a process that can be scheduled independently.

---
Usage: 39 prompt tokens, 38 completion tokens.


## 4. Proof that the model has no memory

Two **separate** requests. The second one does not include the first, so the model cannot know the name.

In [4]:
import chat_loop

def ask_alone(text):
    r = chat_loop.client.chat.completions.create(
        model=chat_loop.MODEL,
        temperature=0.0,
        messages=[{"role": "user", "content": text}],
    )
    return r.choices[0].message.content

print("Request 1:", ask_alone("My project name is Project Chimera."))
print()
print("Request 2:", ask_alone("What is the name of my project?"))

Request 1: Hello! It sounds like you're working on a project called Project Chimera. Is there anything specific you'd like help with related to this project?



Request 2: I'm sorry, but I don't have any information about your project. To find out the name of your project, you would need to ask the person or organization that owns or manages the project.


## 5. The assignment: chat loop with memory (verification test)

`chat_loop.chat()` keeps a `history` list and sends all of it every turn.
Here it reads from a fixed script instead of the keyboard, so the notebook can run by itself.

**Pass:** the last answer says *Project Chimera*.

In [5]:
script = iter([
    "My project name is Project Chimera.",
    "What is the name of my project?",
    "exit",
])

def fake_keyboard(prompt):
    text = next(script)
    print(prompt + text)  # show what was "typed"
    return text

history = chat_loop.chat(read_input=fake_keyboard)

last_answer = history[-1]["content"]
print("\nVERIFICATION:", "PASS" if "chimera" in last_answer.lower() else "FAIL")

Chat started. Type 'exit' or 'quit' to stop.
You: My project name is Project Chimera.
Assistant: Great! Project Chimera sounds intriguing. What is the main goal of your project, and what challenges are you facing?
You: What is the name of my project?


Assistant: Your project is called Project Chimera.
You: exit
Bye!

VERIFICATION: PASS


### What was actually sent to the model on the last turn

The whole list: system prompt, then every user and assistant message, in order.

In [6]:
for message in history:
    print(f"{message['role']:>9}: {message['content']}")

   system: You are a friendly academic mentor for a computer science student. Answer in at most 3 short sentences.
     user: My project name is Project Chimera.
assistant: Great! Project Chimera sounds intriguing. What is the main goal of your project, and what challenges are you facing?
     user: What is the name of my project?
assistant: Your project is called Project Chimera.


## 6. The price of memory: the prompt grows every turn

Because we resend everything, `prompt_tokens` goes up each turn. Long chats cost more and are slower.

In [7]:
history = [{"role": "system", "content": chat_loop.SYSTEM_PROMPT}]
questions = [
    "My project name is Project Chimera.",
    "It is a study helper for students.",
    "What is the name of my project?",
]
for turn, question in enumerate(questions, start=1):
    history.append({"role": "user", "content": question})
    r = chat_loop.client.chat.completions.create(
        model=chat_loop.MODEL, messages=history, temperature=0.7
    )
    print(f"turn {turn}: sent {len(history)} messages -> prompt_tokens = {r.usage.prompt_tokens}")
    history.append({"role": "assistant", "content": r.choices[0].message.content})

turn 1: sent 2 messages -> prompt_tokens = 42


turn 2: sent 4 messages -> prompt_tokens = 79
turn 3: sent 6 messages -> prompt_tokens = 124


## 7. Ollama vs vLLM

| | Ollama | vLLM |
|---|---|---|
| For | laptops, local development | servers with many users |
| Built on | llama.cpp (C/C++) | Python + PyTorch + CUDA |
| Memory trick | normal KV-cache | PagedAttention (KV-cache in pages) |
| Good at | one fast answer | many requests at once (batching) |
| Hardware | Apple Silicon, CPU, normal GPUs | NVIDIA / AMD data-center GPUs |
| API | `/v1/chat/completions` | `/v1/chat/completions` |

Both speak the same OpenAI protocol, so moving this code to vLLM means changing **only** `base_url`.

## 8. Run the real chat yourself

In a terminal, inside this folder:

```bash
source .venv/bin/activate
python chat_loop.py
```

Type `exit` or `quit` to stop.